# 8-bob. 8-BOB. ZARARLI DASTURLARNI ANIQLASH II: DINAMIK TAHLIL VA CHUQUR O'QITISH
## Amaliy misol: shifrlash xatti-harakatini aniqlash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/08-bob/8-4_shifrlash-xatti-harakatini-aniqlash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 8-bob.*

Quyidagi dasturda CryptoDrop g'oyasining soddalashtirilgan modeli quriladi. Foydalanuvchi papkasida 200 ta matnli hujjat bor. To'rt xil jarayon bu fayllarni o'zgartiradi: to'lov talab qiluvchi dastur, faqat fayl boshini shifrlaydigan "ayyorroq" dastur, oddiy matn muharriri va foydalanuvchining o'zi ishga tushirgan qonuniy shifrlash dasturi. Har bir fayl o'zgarishi uchta belgi bo'yicha baholanadi va har bir bajarilgan belgi uchun jarayonga bir ball qo'shiladi. Jarayonning umumiy bali 15 ga yetganda u to'xtatiladi. O'xshashlik siqish yordamida o'lchanadi: ikki fayl o'xshash bo'lsa, ularni birga siqish alohida siqishdan deyarli ko'p joy egallamaydi. Real ma'lumotlarga zarar yetkazmaslik uchun "shifrlash" tasodifiy baytlar bilan almashtirish orqali modellashtirilgan. Dastur amaliy mashg'ulotning A8 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import math
import random
import zlib
from collections import Counter

tasodif = random.Random(7)

def entropiya(b):
    n = len(b)
    return -sum(c / n * math.log2(c / n) for c in Counter(b).values())

def matnmi(b):
    """Fayl turini juda sodda aniqlash: boshidagi 64 bayt o'qiladigan belgilarmi?"""
    return all(32 <= x < 127 or x in (9, 10, 13) for x in b[:64])

def farq(a, b):
    """Siqish orqali o'xshashlik: 0 ga yaqin - o'xshash, 1 ga yaqin - butunlay boshqa."""
    ca, cb, cab = len(zlib.compress(a)), len(zlib.compress(b)), len(zlib.compress(a + b))
    return (cab - min(ca, cb)) / max(ca, cb)

In [2]:
# 1. Foydalanuvchining 200 ta hujjati (sun'iy matnlar)
sozlar = "hisobot shartnoma reja talaba dars natija loyiha moliya bank xodim yig'ilish".split()
def hujjat():
    return " ".join(tasodif.choice(sozlar) for _ in range(tasodif.randint(400, 1200))).encode()
papka = {f"hujjat_{i}.txt": hujjat() for i in range(200)}

In [3]:
# 2. To'rt xil jarayonning faylga yozish amallari
def tolov_dasturi(eski):            # faylni shifrlaydi
    return bytes(x ^ tasodif.randrange(256) for x in eski)
def qisman_shifrlovchi(eski):       # faqat fayl boshini shifrlaydi (tezroq va sezilmasroq)
    return bytes(x ^ tasodif.randrange(256) for x in eski[:256]) + eski[256:]
def matn_muharriri(eski):           # faylga bir necha jumla qo'shadi
    return eski + b" yangi bo'lim qo'shildi."
def shifrlash_dasturi(eski):        # foydalanuvchi o'zi ishga tushirgan qonuniy shifrlash
    return bytes(x ^ tasodif.randrange(256) for x in eski)

In [4]:
# 3. Xatti-harakat ko'rsatkichlari va ball
def ball(eski, yangi):
    b = 0
    b += entropiya(yangi) - entropiya(eski) > 2       # entropiya keskin oshdimi
    b += matnmi(eski) and not matnmi(yangi)           # fayl turi o'zgardimi
    b += farq(eski, yangi) > 0.9                      # mazmun tanib bo'lmas darajada o'zgardimi
    return b

CHEGARA = 15
for nom, amal, soni in [("To'lov talab qiluvchi dastur", tolov_dasturi, 200),
                        ("Qisman shifrlovchi dastur", qisman_shifrlovchi, 200),
                        ("Matn muharriri", matn_muharriri, 30),
                        ("Qonuniy shifrlash dasturi", shifrlash_dasturi, 200)]:
    jami = 0
    for k, (fayl, eski) in enumerate(list(papka.items())[:soni], start=1):
        jami += ball(eski, amal(eski))
        if jami >= CHEGARA:
            print(f"{nom:<30}: {k}-fayldan keyin to'xtatildi (ball {jami})")
            break
    else:
        print(f"{nom:<30}: {soni} ta faylni o'zgartirdi, signal yo'q (ball {jami})")

To'lov talab qiluvchi dastur  : 5-fayldan keyin to'xtatildi (ball 15)
Qisman shifrlovchi dastur     : 15-fayldan keyin to'xtatildi (ball 15)
Matn muharriri                : 30 ta faylni o'zgartirdi, signal yo'q (ball 0)
Qonuniy shifrlash dasturi     : 5-fayldan keyin to'xtatildi (ball 15)


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
To'lov talab qiluvchi dastur  : 5-fayldan keyin to'xtatildi (ball 15)
Qisman shifrlovchi dastur     : 15-fayldan keyin to'xtatildi (ball 15)
Matn muharriri                : 30 ta faylni o'zgartirdi, signal yo'q (ball 0)
Qonuniy shifrlash dasturi     : 5-fayldan keyin to'xtatildi (ball 15)
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 8-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.